# Narrative Structure of the Popol Wuj


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from ipywidgets import interact
from IPython.display import HTML
from great_tables import GT
import plotly.express as px

In [4]:
import numpy as np
from scipy.spatial import distance
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler((0,1))

In [5]:
# base_path = "/content/drive/Othercomputers/My Mac/Repos/Multepal/pv-narrative/notebooks"
base_path = "/Users/rca2t/Repos/Multepal/pv-narrative/notebooks"

In [6]:
def import_tokens(src_id):
    src_path = f"{base_path}/{src_id}"
    token_file = f"{src_path}/{src_id}-TOKEN.csv"
    TOKEN = pd.read_csv(token_file)
    idx_offset = TOKEN.columns.to_list().index('token_str')
    ohco = TOKEN.columns.to_list()[:idx_offset]
    return TOKEN.set_index(ohco)

def chunk_tokens(TOKEN, chunk_size=150, overlap=20, min_len=50):
    """Split text into overlapping word-level chunks."""
    tokens = TOKEN.term_str.dropna().to_list()
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = " ".join(tokens[i:i + chunk_size])
        if len(chunk.split()) >= min_len:  # drop tiny tail chunks
            chunks.append(chunk)
    return chunks

def get_nmf_topics(nmf_model, tfidf_vectorizer, n_top_words):
    words = tfidf_vectorizer.get_feature_names_out()
    topic_words = {}
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_word_indices = topic.argsort()[: -n_top_words - 1 : -1]
        topic_words[f"Topic {topic_idx}"] = [words[i] for i in top_word_indices]
    return pd.DataFrame(topic_words)

In [7]:
# Format for easy data entry
sources = {
    'quc': ['ajtzibab', 'christenson', 'colop', 'christenson_ximenez','ximenez'],
    'spa': ['recinos'],
    'eng': ['tedlock']
}

# Transform for processing below
SOURCES = {}
for lang in sources:
    for src_id in sources[lang]:
        SOURCES[src_id] = {}
        SOURCES[src_id]['lang'] = lang
        SOURCES[src_id]['label'] = src_id.replace("_", " ").title()
        SOURCES[src_id]['tokens'] = import_tokens(src_id)
        # SOURCES[src_id]['text_len'] = SOURCES[src_id]['tokens'].shape[0]

In [8]:
@interact(
    min_df = (1, 20, 1),
    max_df = (.1, 1, .01),
    src_id = SOURCES.keys(),
    n_topics = (2, 20, 1),
    chunk_size = (100, 2000, 5),
    overlap = (0., .9, .01)
)
def plot_text(
        src_id = 'colop',
        chunk_size = 1000,
        overlap = .9,
        min_df = 5,
        max_df = .35,
        n_topics = 8
    ):

    # Create chunked data
    global chunks
    overlap_int = int(overlap * chunk_size)
    TOKEN = SOURCES[src_id]['tokens']
    chunks = chunk_tokens(TOKEN, chunk_size=chunk_size, overlap=overlap_int)

    # Create Count matrix
    count_engine = TfidfVectorizer(lowercase=True,
        max_df=max_df,
        min_df=min_df,
        strip_accents=None,
        norm='l2')
    X = count_engine.fit_transform(chunks)

    # Create topic model
    topic_engine = NMF(
        n_components=n_topics,
        init='nndsvd',
        max_iter=500
    )
    global THETA
    THETA = pd.DataFrame(topic_engine.fit_transform(X))
    THETA.index.name = 'chunk_id'
    THETA.columns.name = 'topic_id'

    # Get topics
    global TOPICS
    TOPICS = get_nmf_topics(topic_engine, count_engine, 7)

    # Print table of topics
    display(GT(TOPICS.apply(lambda x: ' '.join(x)).to_frame('top_terms').reset_index()))

    # Compute distance of neighboring chunks
    D = pd.concat([THETA.shift(1), THETA.shift(0)], axis=1, keys=['a','b']).dropna()\
        .apply(lambda x: distance.cosine(x.a, x.b), axis=1).to_frame('d')
    D['scaled'] = scaler.fit_transform(D[['d']])

    # fig, axes = plt.subplots(2, 1, figsize=(20,10), sharex=True)
    # sns.heatmap(THETA.T, cmap="YlGnBu", ax=axes[0], cbar=None)
    # plt.title(f"{src_id.replace('_', ' ').title()}", fontdict={'size':20, 'weight':'bold'}, y=1.01)
    # plt.xlabel("Syntagm / Event", fontdict={'size': 16})
    # plt.ylabel("Paradigm / Structure", fontdict={'size': 16})
    # plt.show()
    # DF.plot.bar(ax=axes[1], legend=False)
    # sns.despine(left=True, bottom=True)
    # plt.tight_layout()
    # plt.show()

    # fig1 = px.imshow(THETA.T, height=500, width=1500, aspect='auto', color_continuous_scale="YlGnBu")
    # fig1.update_layout(coloraxis_showscale=False)
    # fig1.show()
    # fig2 = px.bar(D, height=500, width=1500)
    # fig2.show()

    # Plotly figures

    w = 1500
    h = 400
    shared_margin = dict(l=80, r=150, t=20, b=75)
    
    # Heatmap
    fig1 = px.imshow(THETA.T, aspect="auto", color_continuous_scale="YlGnBu", labels=dict(x="Syntagm", y="Paradigm"))
    fig1.update_layout(
        width=w,
        height=h,
        margin=shared_margin,  # right margin reserved even with no colorbar
        coloraxis_showscale=False
    )

    # Bar plot
    fig2 = px.bar(D.scaled)
    fig2.update_layout(
        width=w,
        height=h,
        margin=shared_margin,  # match the heatmap exactly
        showlegend=False,
        # title="Scaled cosine distance between chunks",
    )

    fig1.show()
    fig2.show()



interactive(children=(Dropdown(description='src_id', index=2, options=('ajtzibab', 'christenson', 'colop', 'ch…

In [ ]:
@interact(chunk_id = (0, len(chunks) - 1, 1))
def show_chunk(chunk_id=0):
    display(HTML("<div style='font-family: times; font-size:14pt;'>"+chunks[chunk_id]+"</div>"))

interactive(children=(IntSlider(value=0, description='chunk_id', max=270), Output()), _dom_classes=('widget-in…